In [1]:
import requests
from bs4 import BeautifulSoup
import csv
import time
import os
import re  # We need this for the new spacing cleanup

# 1. Setup Data Storage
csv_filename = 'berita_bencana.csv'
file_exists = os.path.isfile(csv_filename)

# Standard browser headers to prevent getting blocked
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

with open(csv_filename, mode='a', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    
    if not file_exists:
        # Include 'author' in the header columns
        writer.writerow(['No', 'judul_berita', 'author', 'tanggal', 'full_text'])

    article_count = 0
    start_offset = 0 
    limit = 10       
    
    # 2. The Pagination Loop
    while article_count < 2000:
        # --- STEP 1: GET THE MENU (THE API) ---
        api_url = f"https://www.cnnindonesia.com/api/v3/search?query=bencana&idtype=1&start={start_offset}&limit={limit}&isrelevance=1"
        response = requests.get(api_url, headers=headers)
        
        if response.status_code != 200:
            print(f"API Error at offset {start_offset}. Status: {response.status_code}")
            break
            
        data = response.json()
        articles_list = data.get('data', [])
        
        if not articles_list:
            print("No more articles found in search.")
            break
            
        for item in articles_list:
            if article_count >= 2000:
                break
                
            # Extract basic info using the correct JSON keys
            judul = item.get('strjudul', '')
            tanggal = item.get('dtmodifon', '') 
            article_url = item.get('url', '')
            
            # Extract Author (Safely handling the list structure)
            author_list = item.get('author', [])
            author_name = "Unknown"
            if author_list and len(author_list) > 0:
                author_name = author_list[0].get('strnmprofil', 'Unknown')
            
            # --- STEP 2: GET THE MEAL (BEAUTIFULSOUP) ---
            full_text = ""
            if article_url:
                try:
                    article_resp = requests.get(article_url, headers=headers)
                    soup = BeautifulSoup(article_resp.text, 'html.parser')
                    
                    content_div = soup.find('div', class_='detail-text')
                    
                    if content_div:
                        # Rip out the "Lihat Juga" table links 
                        for junk_table in content_div.find_all('table', class_='linksisip'):
                            junk_table.decompose() 
                            
                        # Extract and join paragraphs (Notice the separator=" " fix here)
                        paragraphs = content_div.find_all('p')
                        full_text = " ".join([p.get_text(separator=" ", strip=True) for p in paragraphs])
                        
                        # Clean up the advertisement text
                        full_text = full_text.replace("ADVERTISEMENT SCROLL TO CONTINUE WITH CONTENT", "")
                        
                        # Clean up any weird double/triple spaces created by the extraction
                        full_text = re.sub(r'\s+', ' ', full_text).strip()
                        
                    else:
                        full_text = "Content div not found"
                        
                except Exception as e:
                    full_text = f"Error scraping text: {e}"
            
            # --- STEP 3: SAVE TO CSV ---
            article_count += 1
            # Write the updated row including the author
            writer.writerow([article_count, judul, author_name, tanggal, full_text])
            print(f"Saved [{article_count}/2000]: {judul} | By: {author_name}")
            
            # Polite delay between individual article requests
            time.sleep(1)
            
        # Increase offset for the next page of 10 articles
        start_offset += limit
        
        # Polite delay between API page requests
        time.sleep(1.5)

print("\nData mining successfully completed. All files saved to", csv_filename)

Saved [1/2000]: Aceh Tetapkan Status Siaga Bencana hingga 20 April | By: Feri Agus
Saved [2/2000]: FAO Waspadai Bencana Pangan Global Imbas Gangguan di Selat Hormuz | By: Putri Utami
Saved [3/2000]: Mendagri Tito Sebut Inflasi Bulanan 3 Daerah Terdampak Bencana Membaik | By: Indra Hendriana
Saved [4/2000]: Gajah Dikerahkan Bantu Bencana, BKSDA Pastikan Prinsip Animal Welfare | By: Yugo Hindarto
Saved [5/2000]: Ada 2.463 Titik Rawan Bencana di Indonesia Jelang Nataru | By: Yugo Hindarto
Saved [6/2000]: Restrukturisasi Kredit Korban Bencana Sumatra Tembus Rp12,58 T | By: Safyra Primadhyta
Saved [7/2000]: Purbaya Bakal Hapus Utang Pemda Terdampak Bencana | By: Safyra Primadhyta
Saved [8/2000]: BNI Tegaskan Komitmen Hadir Bagi Masyarakat di Tengah Bencana Sumatra | By: Bowie Haryanto
Saved [9/2000]: Kepulauan Sitaro Tetapkan Status Tanggap Darurat Bencana | By: Gilang Fauzi
Saved [10/2000]: Bencana Banjir di Cirebon, 24 Desa Kena Dampak | By: Gilang Fauzi
Saved [11/2000]: Prabowo soal Benc